# PageRank on the npm Dependency Graph

> **Nub Labs Playbook** — Interactive walkthrough: [nublabs.com/case-studies/pagerank](https://www.nublabs.com/case-studies/pagerank)

**The technique:** Power iteration PageRank on a directed dependency graph.  
**The dataset:** 58 npm packages, 59 dependency edges (2024 snapshot).  
**The real application:** Apply the same algorithm to your own graphs — microservice dependencies, CVE blast radius, data lineage, supply chain risk.

Running every cell should give:
- `js-tokens` ranked #1 — not React, not Next.js
- Convergence at iteration 17
- Sum of all ranks = 1.000000

**No pip installs needed** — pure Python, stdlib only (+ matplotlib for charts).

## 1. Dataset

Hand-curated from npm registry metadata (2024 snapshot).  
Each edge `(A, B)` means *"A depends on B"* — rank flows from A → B.

In [ ]:
# (id, weekly_downloads, description)
NODES = [
    # ── Foundation tier — no npm dependencies ────────────────────────────────
    ("lodash",           48_000_000, "JavaScript utility library"),
    ("typescript",       26_000_000, "TypeScript language compiler"),
    ("prettier",         16_000_000, "Opinionated code formatter"),
    ("commander",        24_000_000, "Node.js CLI framework"),
    ("dotenv",           21_000_000, "Environment variable loader"),
    ("uuid",             19_000_000, "UUID/GUID generator"),
    ("semver",           31_000_000, "Semantic versioning parser"),
    ("minimist",         29_000_000, "CLI argument parser"),
    ("ms",               35_000_000, "Millisecond time converter"),
    ("js-tokens",        32_000_000, "JavaScript tokenizer"),
    ("color-name",       28_000_000, "CSS color name database"),
    ("object-assign",    26_000_000, "Object.assign polyfill"),
    ("wrappy",           28_000_000, "Callback wrapping utility"),
    ("inherits",         27_000_000, "Node.js OOP inheritance"),
    ("balanced-match",   30_000_000, "Brace bracket matcher"),
    ("has-flag",         28_000_000, "CLI flag detector"),
    ("path-is-absolute", 27_000_000, "Absolute path checker"),
    ("isexe",            26_000_000, "Executable file check"),
    ("path-key",         25_000_000, "PATH environment key"),
    ("shebang-regex",    24_000_000, "Shebang line regex"),
    ("asynckit",         20_000_000, "Async iteration toolkit"),
    ("delayed-stream",   19_000_000, "Delayed stream wrapper"),
    ("mime-db",          22_000_000, "MIME type database"),
    ("proxy-from-env",   12_000_000, "Proxy URL from env vars"),
    ("concat-map",       29_000_000, "Array concat-map utility"),
    ("fs.realpath",      26_000_000, "fs.realpath implementation"),
    ("react-is",         22_000_000, "React element type check"),
    ("classnames",       15_000_000, "CSS class name utility"),
    ("moment",           13_000_000, "Date manipulation library"),
    # ── Utility tier ─────────────────────────────────────────────────────────
    ("debug",            30_000_000, "Tiny JavaScript debugger"),
    ("loose-envify",     28_000_000, "env.NODE_ENV string replace"),
    ("chalk",            27_000_000, "Terminal string styling"),
    ("ansi-styles",      27_000_000, "ANSI terminal color codes"),
    ("color-convert",    26_000_000, "Color format converter"),
    ("supports-color",   27_000_000, "Terminal color detector"),
    ("mime-types",       24_000_000, "MIME type lookups"),
    ("combined-stream",  20_000_000, "Combined stream API"),
    ("shebang-command",  23_000_000, "Shebang command extractor"),
    ("scheduler",        25_000_000, "React cooperative scheduler"),
    ("once",             27_000_000, "One-time function wrapper"),
    # ── Mid-level tier ───────────────────────────────────────────────────────
    ("follow-redirects", 18_000_000, "HTTP redirect follower"),
    ("prop-types",       18_000_000, "React prop type validation"),
    ("brace-expansion",  29_000_000, "Bash brace expansion"),
    ("which",            22_000_000, "Binary path resolver"),
    ("cross-spawn",      24_000_000, "Cross-platform child_process"),
    ("form-data",        20_000_000, "FormData for Node.js"),
    ("inflight",         24_000_000, "In-flight request dedup"),
    # ── Higher-level tier ────────────────────────────────────────────────────
    ("react",            20_000_000, "React UI library core"),
    ("minimatch",        28_000_000, "Glob pattern matching"),
    ("glob",             25_000_000, "File glob pattern matching"),
    ("react-dom",        19_000_000, "React DOM renderer"),
    # ── Application tier ─────────────────────────────────────────────────────
    ("axios",            18_000_000, "Promise-based HTTP client"),
    ("express",          15_000_000, "Node.js web framework"),
    ("rimraf",           23_000_000, "rm -rf for Node.js"),
    # ── Consumer tier ────────────────────────────────────────────────────────
    ("next",              6_000_000, "The React Framework"),
    ("eslint",           22_000_000, "JavaScript/TypeScript linter"),
    ("webpack",          15_000_000, "JavaScript module bundler"),
    ("jest",             17_000_000, "JavaScript testing framework"),
]

# 59 directed edges: (source, target) means source depends on target
EDGES = [
    ("debug",            "ms"),
    ("chalk",            "ansi-styles"),
    ("chalk",            "supports-color"),
    ("ansi-styles",      "color-convert"),
    ("color-convert",    "color-name"),
    ("supports-color",   "has-flag"),
    ("loose-envify",     "js-tokens"),
    ("scheduler",        "loose-envify"),
    ("react",            "loose-envify"),
    ("react-dom",        "react"),
    ("react-dom",        "loose-envify"),
    ("react-dom",        "scheduler"),
    ("prop-types",       "loose-envify"),
    ("prop-types",       "object-assign"),
    ("prop-types",       "react-is"),
    ("follow-redirects", "debug"),
    ("form-data",        "asynckit"),
    ("form-data",        "combined-stream"),
    ("form-data",        "mime-types"),
    ("combined-stream",  "delayed-stream"),
    ("mime-types",       "mime-db"),
    ("axios",            "follow-redirects"),
    ("axios",            "form-data"),
    ("axios",            "proxy-from-env"),
    ("cross-spawn",      "path-key"),
    ("cross-spawn",      "shebang-command"),
    ("cross-spawn",      "which"),
    ("shebang-command",  "shebang-regex"),
    ("which",            "isexe"),
    ("once",             "wrappy"),
    ("inflight",         "once"),
    ("inflight",         "wrappy"),
    ("brace-expansion",  "balanced-match"),
    ("brace-expansion",  "concat-map"),
    ("minimatch",        "brace-expansion"),
    ("glob",             "fs.realpath"),
    ("glob",             "inflight"),
    ("glob",             "inherits"),
    ("glob",             "minimatch"),
    ("glob",             "once"),
    ("glob",             "path-is-absolute"),
    ("express",          "debug"),
    ("express",          "mime-types"),
    ("rimraf",           "glob"),
    ("next",             "react"),
    ("next",             "react-dom"),
    ("next",             "semver"),
    ("eslint",           "debug"),
    ("eslint",           "glob"),
    ("eslint",           "minimatch"),
    ("eslint",           "chalk"),
    ("eslint",           "semver"),
    ("webpack",          "debug"),
    ("webpack",          "glob"),
    ("webpack",          "semver"),
    ("webpack",          "chalk"),
    ("jest",             "chalk"),
    ("jest",             "glob"),
    ("jest",             "semver"),
]

print(f"Nodes: {len(NODES)}")
print(f"Edges: {len(EDGES)}")

## 2. PageRank Algorithm

Standard power iteration with dangling-node handling.  
Matches the TypeScript implementation at `lib/pagerank/algorithm.ts` in the nublabs repo.

In [ ]:
def pagerank(node_ids, edges, damping=0.85, tolerance=1e-8, max_iter=100):
    """
    Standard PageRank with dangling-node redistribution.

    Parameters
    ----------
    node_ids  : list of str
    edges     : list of (source, target) tuples
    damping   : float  — probability of following a link (Google uses 0.85)
    tolerance : float  — L1-norm convergence threshold
    max_iter  : int    — safety cap on iterations

    Returns
    -------
    ranks        : dict  {node_id: rank}  (all ranks sum to 1.0)
    converged_at : int   iteration number at which Δ < tolerance
    history      : list  rank dicts at each iteration (for plotting)
    """
    N = len(node_ids)
    node_set = set(node_ids)

    out_adj = {n: [] for n in node_ids}
    in_adj  = {n: [] for n in node_ids}
    for src, tgt in edges:
        if src in node_set and tgt in node_set:
            out_adj[src].append(tgt)
            in_adj[tgt].append(src)

    # Dangling nodes: no outgoing edges — their rank is redistributed uniformly
    dangling = [n for n in node_ids if not out_adj[n]]

    ranks   = {n: 1.0 / N for n in node_ids}
    history = [dict(ranks)]
    converged_at = max_iter

    for iteration in range(max_iter):
        dangling_sum = sum(ranks[n] for n in dangling)
        new_ranks = {}
        for n in node_ids:
            link_vote = sum(
                ranks[src] / len(out_adj[src])
                for src in in_adj[n]
            )
            new_ranks[n] = (
                (1 - damping) / N
                + damping * (link_vote + dangling_sum / N)
            )
        delta = sum(abs(new_ranks[n] - ranks[n]) for n in node_ids)
        ranks = new_ranks
        history.append(dict(ranks))
        if delta < tolerance:
            converged_at = iteration + 1
            break

    return ranks, converged_at, history

## 3. Run and Verify

In [ ]:
node_ids = [n[0] for n in NODES]
ranks, converged_at, history = pagerank(node_ids, EDGES)

print(f"Converged at iteration : {converged_at}  (website publishes: 17)")
print(f"Sum of all ranks       : {sum(ranks.values()):.6f}  (should be 1.000000)")
print()

In [ ]:
node_meta = {n[0]: {"downloads": n[1], "description": n[2]} for n in NODES}

# Build in/out degree maps
out_deg = {n: 0 for n in node_ids}
in_deg  = {n: 0 for n in node_ids}
for src, tgt in EDGES:
    out_deg[src] += 1
    in_deg[tgt]  += 1

max_rank = max(ranks.values())

# Print top-15 table
sorted_nodes = sorted(node_ids, key=lambda n: ranks[n], reverse=True)
print(f"{'#':<4} {'Package':<20} {'Rank %':>8}  {'Score/100':>9}  {'In':>4}  {'Out':>4}  Description")
print("-" * 90)
for i, n in enumerate(sorted_nodes[:15], 1):
    score = ranks[n] / max_rank * 100
    desc  = node_meta[n]["description"]
    print(f"{i:<4} {n:<20} {ranks[n]*100:>8.4f}%  {score:>8.1f}   {in_deg[n]:>4}   {out_deg[n]:>4}  {desc}")

## 4. Visualization — Top 15 Bar Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

top15 = sorted_nodes[:15]
scores = [ranks[n] / max_rank * 100 for n in top15]

def bar_color(score):
    if score >= 85: return "#F59E0B"
    if score >= 65: return "#A78BFA"
    if score >= 40: return "#60A5FA"
    if score >= 18: return "#34D399"
    return "#64748B"

colors = [bar_color(s) for s in scores]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#0A0F1E")
ax.set_facecolor("#0A0F1E")

bars = ax.barh(range(len(top15)), scores, color=colors, height=0.7, zorder=3)

ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15, fontfamily="monospace", color="#94A3B8", fontsize=10)
ax.set_xlabel("Relative PageRank score (0–100)", color="#64748B", fontsize=9)
ax.set_title("PageRank — Top 15 npm Packages", color="white", fontsize=13, pad=14)
ax.invert_yaxis()
ax.tick_params(colors="#64748B")
ax.spines[:].set_color("#1E293B")
ax.xaxis.label.set_color("#64748B")
ax.grid(axis="x", color="#1E293B", linewidth=0.7, zorder=0)

# Annotate bars with rank %
for i, (bar, n) in enumerate(zip(bars, top15)):
    ax.text(
        bar.get_width() + 0.5, i,
        f"{ranks[n]*100:.4f}%",
        va="center", color="#64748B", fontsize=8, fontfamily="monospace"
    )

legend_items = [
    mpatches.Patch(color="#F59E0B", label="Very high (≥85)"),
    mpatches.Patch(color="#A78BFA", label="High (65–85)"),
    mpatches.Patch(color="#60A5FA", label="Medium (40–65)"),
    mpatches.Patch(color="#34D399", label="Lower (18–40)"),
    mpatches.Patch(color="#64748B", label="Base (<18)"),
]
ax.legend(handles=legend_items, loc="lower right", facecolor="#1E293B",
          labelcolor="#94A3B8", fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.show()

## 5. Convergence Plot

In [ ]:
# Track how the top-5 packages' ranks evolved per iteration
top5 = sorted_nodes[:5]
palette = ["#F59E0B", "#A78BFA", "#60A5FA", "#34D399", "#F87171"]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#0A0F1E")
ax.set_facecolor("#0A0F1E")

iters = list(range(len(history)))
for pkg, color in zip(top5, palette):
    vals = [h[pkg] * 100 for h in history]
    ax.plot(iters, vals, color=color, linewidth=2, label=pkg)

ax.axvline(converged_at, color="#334155", linestyle="--", linewidth=1)
ax.text(converged_at + 0.3, ax.get_ylim()[1] * 0.95,
        f"converged\niter {converged_at}",
        color="#64748B", fontsize=8)

ax.set_xlabel("Iteration", color="#64748B", fontsize=9)
ax.set_ylabel("PageRank %", color="#64748B", fontsize=9)
ax.set_title("Rank Convergence — Top 5 Packages", color="white", fontsize=13, pad=14)
ax.tick_params(colors="#64748B")
ax.spines[:].set_color("#1E293B")
ax.grid(color="#1E293B", linewidth=0.5, zorder=0)
ax.legend(facecolor="#1E293B", labelcolor="#94A3B8", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()

## 6. The Key Insight — Exclusivity Beats Popularity

> **Why does `js-tokens` rank #1 when only ONE package depends on it?**

The chain:
```
next, react-dom, react, prop-types, scheduler
        │
        ▼
   loose-envify  (4 incoming, 1 outgoing → 100% passthrough)
        │
        ▼
    js-tokens   ← #1
```

`loose-envify` accumulates rank from the entire React ecosystem, then **passes 100% of it to a single target** with no dilution.  
Compare with `glob` (#8): 4 major tools depend on it, but glob splits its rank across **6 outgoing edges**.

**Exclusivity > Popularity.** A sole supplier to one important customer outranks a popular hub with many customers *and* many suppliers.

In [ ]:
# Verify the insight numerically
focus = ["js-tokens", "loose-envify", "glob", "debug", "ms"]
print(f"{'Package':<20}  {'Rank %':>8}  {'In':>4}  {'Out':>4}  Note")
print("-" * 70)
notes = {
    "js-tokens":    "sole dep of loose-envify → 100% passthrough",
    "loose-envify": "4 incoming, 1 outgoing → concentrates React ecosystem rank",
    "glob":         "4 important consumers, BUT 6 outgoing edges → rank diluted",
    "debug":        "4 consumers, 1 outgoing (ms) → debug passes 100% to ms",
    "ms":           "sole dep of debug → receives debug's full rank",
}
for pkg in focus:
    print(f"{pkg:<20}  {ranks[pkg]*100:>8.4f}%  {in_deg[pkg]:>4}  {out_deg[pkg]:>4}  {notes[pkg]}")